<div style="display: flex; justify-content: space-between; align-items: center;">
    <div style="text-align: left; flex: 4;">
        <strong>Author:</strong> Amirhossein Heydari — 
        📧 <a href="mailto:amirhosseinheydari78@gmail.com">amirhosseinheydari78@gmail.com</a> — 
        🐙 <a href="https://github.com/mr-pylin/media-processing-workshop" target="_blank" rel="noopener">github.com/mr-pylin</a>
    </div>
    <div style="display: flex; justify-content: flex-end; flex: 1; gap: 8px; align-items: center; padding: 0;">
        <a href="https://opencv.org/" target="_blank" rel="noopener noreferrer">
            <img src="../assets/external/libraries/opencv/logo/OpenCV_logo_no_text-1.svg"
                 alt="OpenCV Logo"
                 style="max-height: 48px; width: auto;">
        </a>
        <a href="https://pillow.readthedocs.io/" target="_blank" rel="noopener noreferrer">
            <img src="../assets/external/libraries/pillow/logo/pillow-logo-248x250.png"
                 alt="PIL Logo"
                 style="max-height: 48px; width: auto;">
        </a>
        <a href="https://scikit-image.org/" target="_blank" rel="noopener noreferrer">
            <img src="../assets/external/libraries/scikit-image/logo/logo.png"
                 alt="scikit-image Logo"
                 style="max-height: 48px; width: auto;">
        </a>
        <a href="https://scipy.org/" target="_blank" rel="noopener noreferrer">
            <img src="../assets/external/libraries/scipy/logo/logo.svg"
                 alt="SciPy Logo"
                 style="max-height: 48px; width: auto;">
        </a>
    </div>
</div>
<hr>


**Table of contents**<a id='toc0_'></a>    
- [Dependencies](#toc1_)    
- [Load Images](#toc2_)    
- [Common Digital Image Types](#toc3_)    
  - [Binary](#toc3_1_)    
  - [Grayscale](#toc3_2_)    
  - [CMYK](#toc3_3_)    
  - [RGB](#toc3_4_)    
  - [RGBA](#toc3_5_)    
  - [Indexed](#toc3_6_)    
    - [Extract Palette](#toc3_6_1_)    
  - [Multispectral](#toc3_7_)    
    - [False-Color Composites](#toc3_7_1_)    
  - [HDR](#toc3_8_)    
  - [DICOM](#toc3_9_)    
- [Common Pitfalls](#toc4_)    
  - [Color Map](#toc4_1_)    
  - [Minimum and Maximum Values](#toc4_2_)    
  - [BGR Representation](#toc4_3_)    
  - [Loss of Metadata](#toc4_4_)    
- [Hands-on Exercise](#toc5_)    
  - [Bit Depth Adjustment](#toc5_1_)    
  - [Bit Plane Extraction](#toc5_2_)    
  - [Converting Pixels to Physical Dimensions](#toc5_3_)    
  - [Cropping](#toc5_4_)    
    - [Manual](#toc5_4_1_)    
    - [Using PIL](#toc5_4_2_)    
  - [Flipping](#toc5_5_)    
    - [Manual](#toc5_5_1_)    
    - [Using OpenCV](#toc5_5_2_)    
    - [Using PIL](#toc5_5_3_)    
  - [Circular Shift](#toc5_6_)    
    - [Manual](#toc5_6_1_)    
    - [Using NumPy](#toc5_6_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Dependencies](#toc0_)


In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pydicom as pyd
import skimage as ski
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
from numpy.typing import NDArray
from PIL import Image

In [ ]:
# disable automatic figure display (plt.show() required)
# this ensures consistency with .py scripts and gives full control over when plots appear
plt.ioff()

# <a id='toc2_'></a>[Load Images](#toc0_)


In [ ]:
# path to the input image
imp_1 = Path("../assets/original/misc/test_binary.tif")
imp_2 = Path("../assets/external/dip_3rd/CH02/Fig0222(b)(cameraman).tif")
imp_3 = Path("../assets/external/dip_3rd/CH06/edit/FigP0606(color_bars)-cmyk.tif")
imp_4 = Path("../assets/external/dip_3rd/CH06/Fig0638(a)(lenna_RGB).tif")
imp_5 = Path("../assets/external/dip_3rd/CH06/edit/Fig0638(a)(lenna_RGB)-rgba.png")
imp_6 = Path("../assets/external/dip_3rd/CH06/edit/Fig0638(a)(lenna_RGB)-indexed-rgba.png")
imp_7 = sorted(Path("../assets/external/dip_3rd/CH01").glob("Fig0110(*)(WashingtonDC Band*).tif"))
imp_8 = Path("../assets/external/third_party/filesamples.com/sample_hdr.hdr")
imp_9 = Path("../assets/external/third_party/rubomedical.com/0002.DCM")

In [ ]:
# read as PIL images
img_1 = Image.open(imp_1)
img_2 = Image.open(imp_2)
img_3 = Image.open(imp_3)
img_4 = Image.open(imp_4)
img_5 = Image.open(imp_5)
img_6 = Image.open(imp_6)

# read as NDArray
img_7 = np.stack([plt.imread(im) for im in imp_7], axis=-1)
img_8 = cv2.cvtColor(cv2.imread(imp_8, flags=cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB)

# read as pydicom.dataset.FileDataset
img_9 = pyd.dcmread(imp_9)

In [ ]:
# convert PIL.Image to np.ndarray
img_2_np = np.asarray(img_2)
img_4_np = np.asarray(img_4)
img_6_np = np.asarray(img_6)

# <a id='toc3_'></a>[Common Digital Image Types](#toc0_)

- Digital images can be represented in different ways depending on what kind of information they store.
- intensity only, color channels, spectral bands, transparency, etc.

📝 **Docs**:

- The Image Class: [pillow.readthedocs.io/en/stable/reference/Image.html#the-image-class](https://pillow.readthedocs.io/en/stable/reference/Image.html#the-image-class)
- image Attributes: [pillow.readthedocs.io/en/stable/reference/Image.html#image-attributes](https://pillow.readthedocs.io/en/stable/reference/Image.html#image-attributes)


## <a id='toc3_1_'></a>[Binary](#toc0_)

- A **binary image** contains only two possible values per pixel (commonly **0** and **1**, or **0** and **255**).
- Often stored as an **8-bit image** where values are **{0, 255}** even though **logically** it’s **1 bit**.
- It’s typically used for:
  - thresholded images (segmentation),
  - masks (foreground/background),
  - simple detection tasks.


In [ ]:
# plot
plt.figure(figsize=(5, 5), layout="compressed")
plt.title(imp_1.name)
im = plt.imshow(img_1, cmap="gray", vmin=0, vmax=1)
plt.colorbar(im, location="right")
plt.show()

In [ ]:
# log
print(f"type(img_1)                    : {type(img_1)}")
print(f"img_1.format                   : {img_1.format}")
print(f"img_1.mode                     : {img_1.mode}")
print(f"img_1.size                     : {img_1.size}")
print(f"img_1.has_transparency_data    : {img_1.has_transparency_data}")
print(f"img_1.getbands()               : {img_1.getbands()}")
print(f"img_1.getextrema()             : {img_1.getextrema()}")
print(f"img_1.palette                  : {img_1.palette}")
print(f"img_1.info.get('transparency') : {img_1.info.get("transparency")}")
print(f"img_1.info.get('compression')  : {img_1.info.get("compression")}")
print(f"img_1.info.get('dpi')          : {img_1.info.get("dpi")}")
print(f"size                           : {imp_1.stat().st_size / (8 * 1024):.3f} KiB")

## <a id='toc3_2_'></a>[Grayscale](#toc0_)

- A **grayscale image** stores one intensity value per pixel **(no color)**. Pixel values represent **brightness**.
- Many DIP techniques **(filtering, edges)** are often first explained on grayscale.
- Color images can be converted to grayscale for **simplification**.


In [ ]:
# plot
plt.figure(figsize=(5, 5), layout="compressed")
plt.title(imp_2.name)
im = plt.imshow(img_2, cmap="gray", vmin=0, vmax=255)
plt.colorbar(im, location="right")
plt.show()

In [ ]:
# log
print(f"type(img_2)                    : {type(img_2)}")
print(f"img_2.format                   : {img_2.format}")
print(f"img_2.mode                     : {img_2.mode}")
print(f"img_2.size                     : {img_2.size}")
print(f"img_2.has_transparency_data    : {img_2.has_transparency_data}")
print(f"img_2.getbands()               : {img_2.getbands()}")
print(f"img_2.getextrema()             : {img_2.getextrema()}")
print(f"img_2.palette                  : {img_2.palette}")
print(f"img_2.info.get('transparency') : {img_2.info.get("transparency")}")
print(f"img_2.info.get('compression')  : {img_2.info.get("compression")}")
print(f"img_2.info.get('dpi')          : {img_2.info.get("dpi")}")
print(f"size                           : {imp_2.stat().st_size / (8 * 1024):.3f} KiB")

## <a id='toc3_3_'></a>[CMYK](#toc0_)

- **CMYK** is a color model designed for **printing**.
- Each pixel stores **four** channel values as **Cyan**, **Magenta**, **Yellow**, **Key** (black/contrast).
- Usually used with **ink/printing** workflows, not typical **computer screen display**.

✍️ **Notes**:

- Converting from **RGB** (screen) to **CMYK** (print) depends on **printer profiles** and **color management**.
- CMYK can look **different** across devices.


In [ ]:
# separate cmyk channels
img_3_c, img_3_m, img_3_y, img_3_k = img_3.split()

In [ ]:
# plot
fig = plt.figure(figsize=(16, 8), layout="constrained")
gs = GridSpec(nrows=2, ncols=4, figure=fig)

ax1 = fig.add_subplot(gs[0, :])
ax1.imshow(img_3, vmin=0, vmax=1)
ax1.set_title(imp_3.name)

ax2 = fig.add_subplot(gs[1, 0])
im2 = ax2.imshow(img_3_c, cmap="gray", vmin=0, vmax=255)
ax2.set_title("cyan channel")

ax3 = fig.add_subplot(gs[1, 1])
im3 = ax3.imshow(img_3_m, cmap="gray", vmin=0, vmax=255)
ax3.set_title("magenta channel")

ax4 = fig.add_subplot(gs[1, 2])
im4 = ax4.imshow(img_3_y, cmap="gray", vmin=0, vmax=255)
ax4.set_title("yellow channel")

ax5 = fig.add_subplot(gs[1, 3])
im5 = ax5.imshow(img_3_k, cmap="gray", vmin=0, vmax=255)
ax5.set_title("key/black channel")

for ax in fig.axes[1:]:
    ax.set(xticks=[], yticks=[])

fig.colorbar(im2, ax=ax2, location="bottom")
fig.colorbar(im3, ax=ax3, location="bottom")
fig.colorbar(im4, ax=ax4, location="bottom")
fig.colorbar(im5, ax=ax5, location="bottom")

plt.show()

In [ ]:
# log
print(f"type(img_3)                    : {type(img_3)}")
print(f"img_3.format                   : {img_3.format}")
print(f"img_3.mode                     : {img_3.mode}")
print(f"img_3.size                     : {img_3.size}")
print(f"img_3.has_transparency_data    : {img_3.has_transparency_data}")
print(f"img_3.getbands()               : {img_3.getbands()}")
print(f"img_3.getextrema()             : {img_3.getextrema()}")
print(f"img_3.palette                  : {img_3.palette}")
print(f"img_3.info.get('transparency') : {img_3.info.get("transparency")}")
print(f"img_3.info.get('compression')  : {img_3.info.get("compression")}")
print(f"img_3.info.get('dpi')          : {img_3.info.get("dpi")}")
print(f"size                           : {imp_3.stat().st_size / (8 * 1024):.3f} KiB")

## <a id='toc3_4_'></a>[RGB](#toc0_)

- **RGB** (Red, Green, Blue) is the most common digital color model for **screens**.
- In **8-bit** images, each channel typically ranges **0–255** which produces **~16.7 million** possible colors.
- RGB is **device-dependent** (what **“red”** means depends on display characteristics).


In [ ]:
# separate rgb channels
img_4_r, img_4_g, img_4_b = img_4.split()

In [ ]:
# plot
fig = plt.figure(figsize=(16, 10), layout="constrained")
gs = GridSpec(nrows=2, ncols=4, figure=fig)

ax1 = fig.add_subplot(gs[:, 0])
ax1.imshow(img_4, vmin=0, vmax=255)
ax1.set_title(imp_4.name)

ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.imshow(img_4_r, cmap="Reds", vmin=0, vmax=255)
ax2.set_title("Red")

ax3 = fig.add_subplot(gs[0, 2])
im3 = ax3.imshow(img_4_g, cmap="Greens", vmin=0, vmax=255)
ax3.set_title("Green")

ax4 = fig.add_subplot(gs[0, 3])
im4 = ax4.imshow(img_4_b, cmap="Blues", vmin=0, vmax=255)
ax4.set_title("Blue")

ax5 = fig.add_subplot(gs[1, 1])
im5 = ax5.imshow(img_4_r, cmap="gray", vmin=0, vmax=255)
ax5.set_title("Red [Gray]")

ax6 = fig.add_subplot(gs[1, 2])
im6 = ax6.imshow(img_4_g, cmap="gray", vmin=0, vmax=255)
ax6.set_title("Green [Gray]")

ax7 = fig.add_subplot(gs[1, 3])
im7 = ax7.imshow(img_4_b, cmap="gray", vmin=0, vmax=255)
ax7.set_title("Blue [Gray]")

for ax in fig.axes[1:]:
    ax.set(xticks=[], yticks=[])

fig.colorbar(im2, ax=ax2, location="top", label="R")
fig.colorbar(im3, ax=ax3, location="top", label="G")
fig.colorbar(im4, ax=ax4, location="top", label="B")
fig.colorbar(im5, ax=ax5, location="bottom", label="R")
fig.colorbar(im6, ax=ax6, location="bottom", label="G")
fig.colorbar(im7, ax=ax7, location="bottom", label="B")

plt.show()

In [ ]:
# log
print(f"type(img_4)                    : {type(img_4)}")
print(f"img_4.format                   : {img_4.format}")
print(f"img_4.mode                     : {img_4.mode}")
print(f"img_4.size                     : {img_4.size}")
print(f"img_4.has_transparency_data    : {img_4.has_transparency_data}")
print(f"img_4.getbands()               : {img_4.getbands()}")
print(f"img_4.getextrema()             : {img_4.getextrema()}")
print(f"img_4.palette                  : {img_4.palette}")
print(f"img_4.info.get('transparency') : {img_4.info.get("transparency")}")
print(f"img_4.info.get('compression')  : {img_4.info.get("compression")}")
print(f"img_4.info.get('dpi')          : {img_4.info.get("dpi")}")
print(f"size                           : {imp_4.stat().st_size / (8 * 1024):.3f} KiB")

## <a id='toc3_5_'></a>[RGBA](#toc0_)

- **RGBA** extends **RGB** by adding an **alpha channel** which corresponds to **transparency** (opacity control).
- $A=1$ (or 255) is fully **opaque** and $A=0$ is fully **transparent**.


In [ ]:
# separate rgba channels
img_5_r, img_5_g, img_5_b, img_5_a = img_5.split()

In [ ]:
# plot
fig = plt.figure(figsize=(14, 8), layout="constrained")
gs = GridSpec(nrows=2, ncols=4, figure=fig)

ax1 = fig.add_subplot(gs[0, :])
ax1.imshow(img_5, vmin=0, vmax=1)
ax1.set_title(imp_5.name)

ax2 = fig.add_subplot(gs[1, 0])
im2 = ax2.imshow(img_5_r, cmap="Reds", vmin=0, vmax=255)
ax2.set_title("Red")

ax3 = fig.add_subplot(gs[1, 1])
im3 = ax3.imshow(img_5_g, cmap="Greens", vmin=0, vmax=255)
ax3.set_title("Green")

ax4 = fig.add_subplot(gs[1, 2])
im4 = ax4.imshow(img_5_b, cmap="Blues", vmin=0, vmax=255)
ax4.set_title("Blue")

ax5 = fig.add_subplot(gs[1, 3])
im5 = ax5.imshow(img_5_a, cmap="gray", vmin=0, vmax=255)
ax5.set_title("Alpha")

for ax in fig.axes[1:]:
    ax.set(xticks=[], yticks=[])

fig.colorbar(im2, ax=ax2, location="bottom")
fig.colorbar(im3, ax=ax3, location="bottom")
fig.colorbar(im4, ax=ax4, location="bottom")
fig.colorbar(im5, ax=ax5, location="bottom")

plt.show()

In [ ]:
# log
print(f"type(img_5)                    : {type(img_5)}")
print(f"img_5.format                   : {img_5.format}")
print(f"img_5.mode                     : {img_5.mode}")
print(f"img_5.size                     : {img_5.size}")
print(f"img_5.has_transparency_data    : {img_5.has_transparency_data}")
print(f"img_5.getbands()               : {img_5.getbands()}")
print(f"img_5.getextrema()             : {img_5.getextrema()}")
print(f"img_5.palette                  : {img_5.palette}")
print(f"img_5.info.get('transparency') : {img_5.info.get("transparency")}")
print(f"img_5.info.get('compression')  : {img_5.info.get("compression")}")
print(f"img_5.info.get('dpi')          : {img_5.info.get("dpi")}")
print(f"size                           : {imp_5.stat().st_size / (8 * 1024):.3f} KiB")

## <a id='toc3_6_'></a>[Indexed](#toc0_)

- An **indexed image** stores pixels as **indices** into a **color palette** (lookup table, LUT).
- **Representation:**
  - **Image array:** integer indices (e.g., 0–15)
  - **Palette:** list of colors (e.g., 16 RGB entries)
  - Each pixel value means use `palette[index]`.
- **Benefits:**
  - **Saves memory** when there are few distinct colors.
  - Useful for **certain diagrams**, and some **compression schemes**.


In [ ]:
# separate rgba channels
img_6_r, img_6_g, img_6_b, img_6_a = img_6.convert("RGBA").split()

In [ ]:
# plot
fig = plt.figure(figsize=(14, 8), layout="constrained")
gs = GridSpec(nrows=2, ncols=4, figure=fig)

ax1 = fig.add_subplot(gs[0, :])
ax1.imshow(img_6, vmin=0, vmax=1)
ax1.set_title(imp_6.name)

ax2 = fig.add_subplot(gs[1, 0])
im2 = ax2.imshow(img_6_r, cmap="Reds", vmin=0, vmax=255)
ax2.set_title("Red")

ax3 = fig.add_subplot(gs[1, 1])
im3 = ax3.imshow(img_6_g, cmap="Greens", vmin=0, vmax=255)
ax3.set_title("Green")

ax4 = fig.add_subplot(gs[1, 2])
im4 = ax4.imshow(img_6_b, cmap="Blues", vmin=0, vmax=255)
ax4.set_title("Blue")

ax5 = fig.add_subplot(gs[1, 3])
im5 = ax5.imshow(img_6_a, cmap="gray", vmin=0, vmax=255)
ax5.set_title("Alpha")

for ax in fig.axes[1:]:
    ax.set(xticks=[], yticks=[])

fig.colorbar(im2, ax=ax2, location="bottom")
fig.colorbar(im3, ax=ax3, location="bottom")
fig.colorbar(im4, ax=ax4, location="bottom")
fig.colorbar(im5, ax=ax5, location="bottom")

plt.show()

In [ ]:
# log
print(f"type(img_6)                    : {type(img_6)}")
print(f"img_6.format                   : {img_6.format}")
print(f"img_6.mode                     : {img_6.mode}")
print(f"img_6.size                     : {img_6.size}")
print(f"img_6.has_transparency_data    : {img_6.has_transparency_data}")
print(f"img_6.getbands()               : {img_6.getbands()}")
print(f"img_6.getextrema()             : {img_6.getextrema()}")
print(f"img_6.palette                  : {img_6.palette}")
print(f"img_6.info.get('transparency') : {img_6.info.get("transparency")}")
print(f"img_6.info.get('compression')  : {img_6.info.get("compression")}")
print(f"img_6.info.get('dpi')          : {img_6.info.get("dpi")}")
print(f"size                           : {imp_6.stat().st_size / (8 * 1024):.3f} KiB")

### <a id='toc3_6_1_'></a>[Extract Palette](#toc0_)


In [ ]:
img_6_palette = np.array(img_6.getpalette(), dtype=np.uint8).reshape(-1, 3)

# log
print(f"colors.shape : {img_6_palette.shape}")
print(f"colors :\n{img_6_palette}")

In [ ]:
img_6_opacity = np.ones(len(img_6_palette), dtype=np.uint8)
img_6_opacity[img_6.info["transparency"]] = 0

# log
print(img_6_opacity)

In [ ]:
img_6_colors = np.concatenate([img_6_palette / 255, img_6_opacity[:, None]], axis=1)

# plot
plt.imshow(img_6_np, cmap=ListedColormap(img_6_colors), vmin=0, vmax=len(img_6_palette))
plt.title(imp_6.name)
plt.axis("off")
plt.show()

## <a id='toc3_7_'></a>[Multispectral](#toc0_)

- **Multispectral** images contain **multiple spectral bands**, not just RGB.
- Each band is a measurement of the scene **reflectance** (or radiance) at a specific **wavelength range**.
- Multispectral preserves more detailed spectral signatures rather than RGB.

✍️ **Applications:**

- Agriculture (crop health)
- Remote sensing / land cover
- Material identification
- Medical imaging (in some setups)


In [ ]:
wavelengths = [
    "0.45-0.52 μm",  # Visible Blue
    "0.53-0.61 μm",  # Visible Green
    "0.63-0.69 μm",  # Visible Red
    "0.78-0.90 μm",  # Near infrared
    "1.55-1.75 μm",  # Middle infrared
    "10.4-12.5 μm",  # Thermal infrared
    "2.09-2.35 μm",  # Short-wave infrared
]

In [ ]:
# plot
fig, axs = plt.subplots(1, 7, figsize=(16, 5), layout="compressed")
fig.suptitle("Multispectral image: Washington DC")
for i, ax in enumerate(axs):
    ax.imshow(img_7[:, :, i], cmap="gray")
    ax.set_title(wavelengths[i])
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

In [ ]:
# log
print(f"type(img_7): {type(img_7)}")
print(f"img_7.shape: {img_7.shape}")
print(f"img_7.dtype: {img_7.dtype}")

### <a id='toc3_7_1_'></a>[False-Color Composites](#toc0_)

- **False-color composites** allow visualization of multispectral images by mapping **three selected bands** to the **Red, Green, and Blue (RGB)** display channels.
- This technique creates an **artificial representation** to highlight specific features like vegetation, water, and urban areas.

✍️ **Key Principles:**
- **Flexibility:** Any **3-band combination** can be chosen depending on the features to highlight.
- **Visualization:** For better understanding, individual band contributions can be visualized using **grayscale plots with colorbars**.

📊 **Regions of Interest:**

| Feature | **Band Mapping (RGB)** | **Wavelength & Band Reference** |
| :--- | :--- | :--- |
| **Vegetation Detection** | **Red** → NIR<br>**Green** → Red<br>**Blue** → Green | (0.78–0.90 μm, Band 4)<br>(0.63–0.69 μm, Band 3)<br>(0.53–0.61 μm, Band 2) |
| **Water Bodies** | **Red** → Green<br>**Green** → Blue<br>**Blue** → NIR | (0.53–0.61 μm, Band 2)<br>(0.45–0.52 μm, Band 1)<br>(0.78–0.90 μm, Band 4) |
| **Urban / Built-Up** | **Red** → MIR<br>**Green** → NIR<br>**Blue** → Red | (1.55–1.75 μm, Band 5)<br>(0.78–0.90 μm, Band 4)<br>(0.63–0.69 μm, Band 3) |


In [ ]:
# vegetation index (NIR-Red-Green) -> RGB
img_7_vegetation = np.dstack(
    (
        img_7[:, :, 3],  # band 4 -> NIR -> Red
        img_7[:, :, 2],  # band 3 -> Red -> Green
        img_7[:, :, 1],  # band 2 -> Green -> Blue
    )
)

In [ ]:
# water index (Green-Blue-NIR) -> RGB
img_7_water = np.dstack(
    (
        img_7[:, :, 1],  # band 2 -> Green -> Red
        img_7[:, :, 0],  # band 1 -> Blue -> Green
        img_7[:, :, 3],  # band 4 -> NIR -> Blue
    )
)

In [ ]:
# urban index (MIR-NIR-Red) -> RGB
img_7_urban = np.dstack(
    (
        img_7[:, :, 4],  # band 5 -> MIR -> Red
        img_7[:, :, 3],  # band 4 -> NIR -> Green
        img_7[:, :, 2],  # band 3 -> Red -> Blue
    )
)

In [ ]:
# plot
images = [img_7_vegetation, img_7_water, img_7_urban]
titles = ["Vegetation index", "Water index", "Urban index"]
images_gray = [ski.color.rgb2gray(img / img.max()) for img in images]

fig, axs = plt.subplots(2, 3, figsize=(13, 8), layout="compressed")
fig.suptitle("Multispectral image: Washington DC")

# plot RGB false-color composites
for ax, img, title in zip(axs[0], images, titles):
    ax.imshow(img / img.max())
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

# plot grayscale versions
for ax, img, title in zip(axs[1], images_gray, titles):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{title} - Grayscale")
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()

## <a id='toc3_8_'></a>[HDR](#toc0_)

- **HDR** (High Dynamic Range) images store a **wider range** of brightness values than **standard 8-bit SDR** images.
- **SDR** typically limits brightness to **0–255** (or **0–1** normalized).
- **HDR** can preserve detail in both **very dark** and **very bright** areas.

📝 **Docs**:

- Image Source: [filesamples.com/formats/hdr](https://filesamples.com/formats/hdr)  


In [ ]:
# plot
fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(16, 5), layout="compressed")

axs[0].imshow(img_8, vmin=0, vmax=1)
axs[0].set_title(imp_8.name)

im2 = axs[1].imshow(img_8[:, :, 0], cmap="gray", vmin=0, vmax=1)
axs[1].set_title("Red")

im3 = axs[2].imshow(img_8[:, :, 1], cmap="gray", vmin=0, vmax=1)
axs[2].set_title("Green")

im4 = axs[3].imshow(img_8[:, :, 2], cmap="gray", vmin=0, vmax=1)
axs[3].set_title("Blue")


for ax in fig.axes[1:]:
    ax.set(xticks=[], yticks=[])

fig.colorbar(im2, ax=axs[1], location="bottom")
fig.colorbar(im3, ax=axs[2], location="bottom")
fig.colorbar(im4, ax=axs[3], location="bottom")

plt.show()

In [ ]:
# log
print(f"type(img_8): {type(img_8)}")
print(f"img_8.shape: {img_8.shape}")
print(f"img_8.dtype: {img_8.dtype}")

## <a id='toc3_9_'></a>[DICOM](#toc0_)

- **DICOM** (Digital Imaging and Communications in Medicine) is a **medical imaging standard** used for storing and transmitting medical scans (e.g., **MRI**, **CT**).
- It includes **pixel data** (image) and **extensive metadata** (patient, modality, acquisition settings, spatial information, etc.).

📝 **See Also**:

- Medical imaging: [en.wikipedia.org/wiki/Medical_imaging](https://en.wikipedia.org/wiki/Medical_imaging)
- Image Source: [rubomedical.com/dicom_files](https://www.rubomedical.com/dicom_files/)


In [ ]:
img_9_np = img_9.pixel_array

In [ ]:
# log
print(f"type(img_9_np): {type(img_9_np)}")
print(f"img_9_np.shape: {img_9_np.shape}")
print(f"img_9_np.dtype: {img_9_np.dtype}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4), layout="compressed")


def update(i):
    ax.clear()
    ax.imshow(img_9_np[i], cmap="gray")
    ax.set_title(f"Frame {i}")
    ax.axis("off")


anim = FuncAnimation(fig, update, frames=img_9_np.shape[0], interval=1000 / 30)
HTML(anim.to_jshtml())

# <a id='toc4_'></a>[Common Pitfalls](#toc0_)


## <a id='toc4_1_'></a>[Color Map](#toc0_)


In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(8, 4), layout="compressed")
axs[0].imshow(img_2)
axs[0].set(title="Wrong cmap (pesudo color)", xticks=[], yticks=[])
axs[1].imshow(img_2, cmap="gray")
axs[1].set(title="Correct cmap", xticks=[], yticks=[])
plt.show()

## <a id='toc4_2_'></a>[Minimum and Maximum Values](#toc0_)


In [ ]:
img_9 = np.zeros((256, 256), dtype=np.uint8)
img_9[:, :128] = 64
img_9[:, 128:] = 196

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(8, 4), layout="compressed")
axs[0].imshow(img_9, cmap="gray")
axs[0].set(title="Normalized", xticks=[], yticks=[])
axs[1].imshow(img_9, cmap="gray", vmin=0, vmax=255)
axs[1].set(title="Real", xticks=[], yticks=[])
plt.show()

## <a id='toc4_3_'></a>[BGR Representation](#toc0_)


In [ ]:
img_10 = cv2.imread(imp_4, flags=cv2.IMREAD_COLOR)  # default order is BGR

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(8, 4), layout="compressed")
axs[0].imshow(img_10)
axs[0].set(title="Swapped Channels", xticks=[], yticks=[])
axs[1].imshow(img_10[:, :, ::-1])
axs[1].set(title="Correct Channels", xticks=[], yticks=[])
plt.show()

## <a id='toc4_4_'></a>[Loss of Metadata](#toc0_)


In [ ]:
img_11 = np.array(Image.open(imp_6))  # wrong conversion

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(8, 4), layout="compressed")
axs[0].imshow(img_11)
axs[0].set(title="Lost Metadata", xticks=[], yticks=[])
axs[1].imshow(img_6)
axs[1].set(title="Correct Image", xticks=[], yticks=[])
plt.show()

# <a id='toc5_'></a>[Hands-on Exercise](#toc0_)


## <a id='toc5_1_'></a>[Bit Depth Adjustment](#toc0_)


In [ ]:
img_2_8bit = img_2_np
img_2_4bit = np.astype(np.round((img_2_np / 255) * (2**4 - 1)), np.uint8)
img_2_2bit = np.astype(np.round((img_2_np / 255) * (2**2 - 1)), np.uint8)
img_2_1bit = np.astype(np.round((img_2_np / 255) * (2**1 - 1)), np.uint8)

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=4, figsize=(16, 8), layout="compressed")

axs[0, 0].imshow(img_2_8bit, cmap="gray")
axs[0, 0].set_title("8bpp")
axs[0, 1].imshow(img_2_4bit, cmap="gray")
axs[0, 1].set_title("4bpp")
axs[0, 2].imshow(img_2_2bit, cmap="gray")
axs[0, 2].set_title("2bpp")
axs[0, 3].imshow(img_2_1bit, cmap="gray")
axs[0, 3].set_title("1bpp")

axs[1, 0].imshow(img_2_8bit[40:80, 100:140], cmap="gray")
axs[1, 1].imshow(img_2_4bit[40:80, 100:140], cmap="gray")
axs[1, 2].imshow(img_2_2bit[40:80, 100:140], cmap="gray")
axs[1, 3].imshow(img_2_1bit[40:80, 100:140], cmap="gray")

plt.show()

## <a id='toc5_2_'></a>[Bit Plane Extraction](#toc0_)

- Bit-plane extraction is a technique used in image processing to visualize and analyze the significance of individual bits in an image.


In [ ]:
img_2_bit_planes = []
for i in range(8):
    bit_plane = (img_2_np >> i) & 1  # shift and mask
    img_2_bit_planes.append(bit_plane * 255)

In [ ]:
# plot
fig, axs = plt.subplots(1, 8, figsize=(24, 3), layout="compressed")
for i, ax in enumerate(axs.flat):
    ax.imshow(img_2_bit_planes[7 - i], cmap="gray")
    ax.set_title(f"Bit {7 - i}")
    ax.axis("off")
plt.show()

## <a id='toc5_3_'></a>[Converting Pixels to Physical Dimensions](#toc0_)

- When working with images, we often need to switch between **digital units (pixels)** and **physical units (inches or centimeters)**.  
- This conversion is essential for tasks like cropping by real-world size, printing images at the right dimensions, or resizing images for publications.  

✍ **Key Concepts**:

- **Pixel dimensions**: The width and height of an image in pixels (e.g., 1920×1080 px).  
- **Physical dimensions**: The size of the image in real-world units (inches, cm, mm).  
- **DPI (dots per inch) / PPI (pixels per inch)**: The scaling factor that links pixels to physical size.  

🔢 **Converting Between Pixels and Inches/cm**:

- $1$ inch = $2.54$ cm

$$
\text{pixels} = \text{inches} \times \text{DPI}
$$



In [ ]:
x_px, y_px = img_2.size  # width, height pixels
x_dpi, y_dpi = img_2.info["dpi"]  # horizontal, vertical DPI

# log
print(f"x_px  : {x_px} px")
print(f"y_px  : {y_px} px")
print(f"x_dpi : {x_dpi:.0f} dpi")
print(f"y_dpi : {y_dpi:.0f} dpi")

In [ ]:
# pixel to inch
x_px2in = x_px / x_dpi
y_px2in = y_px / y_dpi

# pixel to cm
x_px2cm = x_px2in * 2.54
y_px2cm = y_px2in * 2.54

# log
print(f"x_px2in : {x_px2in:.2f} in")
print(f"y_px2in : {y_px2in:.2f} in")
print(f"x_px2cm : {x_px2cm:.2f} cm")
print(f"y_px2cm : {y_px2cm:.2f} cm")

## <a id='toc5_4_'></a>[Cropping](#toc0_)

- Cropping an image involves selecting a **Region of Interest (RoI)** and discarding the rest.


### <a id='toc5_4_1_'></a>[Manual](#toc0_)


In [ ]:
def crop(image: NDArray, x_start: int, x_end: int, y_start: int, y_end: int) -> NDArray:
    return image[y_start:y_end, x_start:x_end]

In [ ]:
img_2_crop_1 = crop(img_2_np, 0, 128, 0, 128)
img_2_crop_2 = crop(img_2_np, 64, 192, 64, 192)
img_2_crop_3 = crop(img_2_np, 100, 150, 50, 100)
img_4_crop_1 = crop(img_4_np, 0, 256, 0, 256)
img_4_crop_2 = crop(img_4_np, 120, 300, 120, 300)
img_4_crop_3 = crop(img_4_np, 240, 290, 250, 300)

In [ ]:
# plot
fig, axs = plt.subplots(nrows=2, ncols=4, figsize=(16, 8), layout="compressed")
images = [
    [img_2, img_2_crop_1, img_2_crop_2, img_2_crop_3],
    [img_4, img_4_crop_1, img_4_crop_2, img_4_crop_3],
]
titles = [
    ["img_2", "img_2_np[:128, :128]", "img_2_np[64:192, 64:192]", "img_2_np[50:100, 100:150]"],
    ["img_4", "img_4_np[:256, :256]", "img_4_np[120:300, 120:300]", "img_4_np[250:300, 240:290]"],
]
for i in range(2):
    for j in range(4):
        axs[i, j].imshow(images[i][j], cmap="gray")
        axs[i, j].set_title(titles[i][j], fontdict={"family": "consolas"})
plt.show()

### <a id='toc5_4_2_'></a>[Using PIL](#toc0_)

📝 **Docs**:

- `PIL.Image.Image.crop`: [pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.crop](https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.crop)


In [ ]:
img_2_crop_4 = img_2.crop(box=(0, 0, 128, 128))
img_2_crop_5 = img_2.crop(box=(64, 64, 192, 192))
img_2_crop_6 = img_2.crop(box=(100, 50, 150, 100))
img_4_crop_4 = img_4.crop(box=(0, 0, 256, 256))
img_4_crop_5 = img_4.crop(box=(120, 120, 300, 300))
img_4_crop_6 = img_4.crop(box=(240, 250, 290, 300))

In [ ]:
# plot
fig, axs = plt.subplots(nrows=2, ncols=4, figsize=(16, 8), layout="compressed")
images = [
    [img_2, img_2_crop_1, img_2_crop_2, img_2_crop_3],
    [img_4, img_4_crop_1, img_4_crop_2, img_4_crop_3],
]
titles = [
    ["img_2", "img_2_crop_4", "img_2_crop_5", "img_2_crop_6"],
    ["img_4", "img_4_crop_4", "img_4_crop_5", "img_4_crop_6"],
]
for i in range(2):
    for j in range(4):
        axs[i, j].imshow(images[i][j], cmap="gray")
        axs[i, j].set_title(titles[i][j], fontdict={"family": "consolas"})
plt.show()

## <a id='toc5_5_'></a>[Flipping](#toc0_)


### <a id='toc5_5_1_'></a>[Manual](#toc0_)


In [ ]:
def flip(image: NDArray, axis: int) -> NDArray:
    if axis == 0:
        return image[::-1]
    elif axis == 1:
        return image[:, ::-1]
    elif axis == 2:
        return image[:, :, ::-1]  # reverse the color channels
    else:
        raise ValueError("Axis must be 0, 1, or 2")

In [ ]:
img_2_flip_1 = flip(img_2_np, axis=0)
img_2_flip_2 = flip(img_2_np, axis=1)
img_2_flip_3 = flip(img_2_flip_1, axis=1)
img_4_flip_1 = flip(img_4_np, axis=0)
img_4_flip_2 = flip(img_4_np, axis=1)
img_4_flip_3 = flip(img_4_flip_1, axis=1)

In [ ]:
# plot
fig, axs = plt.subplots(2, 4, figsize=(16, 8), layout="compressed")
images = [
    [img_2, img_2_flip_1, img_2_flip_2, img_2_flip_3],
    [img_4, img_4_flip_1, img_4_flip_2, img_4_flip_3],
]
titles = [
    ["img_2", "img_2_flip_1", "img_2_flip_2", "img_2_flip_3"],
    ["img_4", "img_4_flip_1", "img_4_flip_2", "img_4_flip_3"],
]
for i in range(2):
    for j in range(4):
        axs[i, j].imshow(images[i][j], cmap="gray")
        axs[i, j].set_title(titles[i][j], fontdict={"family": "consolas"})
plt.show()

### <a id='toc5_5_2_'></a>[Using OpenCV](#toc0_)

📝 **Docs**:

- `cv2.flip`: [docs.opencv.org/master/d2/de8/group__core__array.html#gaca7be533e3dac7feb70fc60635adf441](https://docs.opencv.org/master/d2/de8/group__core__array.html#gaca7be533e3dac7feb70fc60635adf441)


In [ ]:
img_2_flip_4 = cv2.flip(img_2_np, 0)
img_2_flip_5 = cv2.flip(img_2_np, 1)
img_2_flip_6 = cv2.flip(img_2_flip_4, 1)
img_4_flip_4 = cv2.flip(img_4_np, 0)
img_4_flip_5 = cv2.flip(img_4_np, 1)
img_4_flip_6 = cv2.flip(img_4_flip_4, 1)

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(16, 8), layout="compressed")
images = [
    [img_2, img_2_flip_4, img_2_flip_5, img_2_flip_6],
    [img_4, img_4_flip_4, img_4_flip_5, img_4_flip_6],
]
titles = [
    ["img_2", "img_2_flip_4", "img_2_flip_5", "img_2_flip_6"],
    ["img_4", "img_4_flip_4", "img_4_flip_5", "img_4_flip_6"],
]
for i in range(2):
    for j in range(4):
        axs[i, j].imshow(images[i][j], cmap="gray")
        axs[i, j].set_title(titles[i][j], fontdict={"family": "consolas"})
plt.show()

### <a id='toc5_5_3_'></a>[Using PIL](#toc0_)

📝 **Docs**:

- `PIL.Image.Image.transpose`: [pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.transpose](https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.transpose)


In [ ]:
img_2_flip_7 = img_2.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
img_2_flip_8 = img_2.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
img_2_flip_9 = img_2_flip_7.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
img_4_flip_7 = img_4.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
img_4_flip_8 = img_4.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
img_4_flip_9 = img_4_flip_7.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

In [ ]:
# plot
fig, axs = plt.subplots(2, 4, figsize=(16, 8), layout="compressed")
images = [
    [img_2, img_2_flip_7, img_2_flip_8, img_2_flip_9],
    [img_4, img_4_flip_7, img_4_flip_8, img_4_flip_9],
]
titles = [
    ["img_2", "img_2_flip_1", "img_2_flip_2", "img_2_flip_3"],
    ["img_4", "img_4_flip_1", "img_4_flip_2", "img_4_flip_3"],
]
for i in range(2):
    for j in range(4):
        axs[i, j].imshow(images[i][j], cmap="gray")
        axs[i, j].set_title(titles[i][j], fontdict={"family": "consolas"})
plt.show()

## <a id='toc5_6_'></a>[Circular Shift](#toc0_)

- Circular shifting moves pixels in a cyclic manner.


### <a id='toc5_6_1_'></a>[Manual](#toc0_)


In [ ]:
def circular_shift(image: NDArray, dx: int, dy: int) -> NDArray:
    x_shift = np.hstack((image[:, dx:], image[:, :dx]))
    y_shift = np.vstack((x_shift[dy:], x_shift[:dy]))
    return y_shift

In [ ]:
img_2_cshift_1 = circular_shift(img_2_np, dx=0, dy=128)
img_2_cshift_2 = circular_shift(img_2_np, dx=128, dy=0)
img_2_cshift_3 = circular_shift(img_2_np, dx=128, dy=128)
img_4_cshift_1 = circular_shift(img_4_np, dx=0, dy=256)
img_4_cshift_2 = circular_shift(img_4_np, dx=256, dy=0)
img_4_cshift_3 = circular_shift(img_4_np, dx=256, dy=256)

In [ ]:
# plot
fig, axs = plt.subplots(2, 4, figsize=(16, 8), layout="compressed")
images = [
    [img_2, img_2_cshift_1, img_2_cshift_2, img_2_cshift_3],
    [img_4, img_4_cshift_1, img_4_cshift_2, img_4_cshift_3],
]
titles = [
    ["img_2", "img_2_cshift_1", "img_2_cshift_2", "img_2_cshift_3"],
    ["img_4", "img_4_cshift_1", "img_4_cshift_2", "img_4_cshift_3"],
]
for i in range(2):
    for j in range(4):
        axs[i, j].imshow(images[i][j], cmap="gray")
        axs[i, j].set_title(titles[i][j], fontdict={"family": "consolas"})
plt.show()

### <a id='toc5_6_2_'></a>[Using NumPy](#toc0_)

📝 **Docs**:

- `numpy.roll`: [numpy.org/doc/stable/reference/generated/numpy.roll.html](https://numpy.org/doc/stable/reference/generated/numpy.roll.html)


In [ ]:
img_2_cshift_4 = np.roll(img_2_np, shift=(128, 0), axis=(0, 1))
img_2_cshift_5 = np.roll(img_2_np, shift=(0, 128), axis=(0, 1))
img_2_cshift_6 = np.roll(img_2_np, shift=(128, 128), axis=(0, 1))
img_4_cshift_4 = np.roll(img_4_np, shift=(256, 0), axis=(0, 1))
img_4_cshift_5 = np.roll(img_4_np, shift=(0, 256), axis=(0, 1))
img_4_cshift_6 = np.roll(img_4_np, shift=(256, 256), axis=(0, 1))

In [ ]:
# plot
fig, axs = plt.subplots(2, 4, figsize=(16, 8), layout="compressed")
images = [
    [img_2, img_2_cshift_4, img_2_cshift_5, img_2_cshift_6],
    [img_4, img_4_cshift_4, img_4_cshift_5, img_4_cshift_6],
]
titles = [
    ["img_2", "img_2_cshift_4", "img_2_cshift_5", "img_2_cshift_6"],
    ["img_4", "img_4_cshift_4", "img_4_cshift_5", "img_4_cshift_6"],
]
for i in range(2):
    for j in range(4):
        axs[i, j].imshow(images[i][j], cmap="gray")
        axs[i, j].set_title(titles[i][j], fontdict={"family": "consolas"})
plt.show()